In [1]:
document = """
Apex Motion's Pune plant manufactures BMS Controller X1.

BMS Controller X1 requires MCU-900.

MCU-900 is supplied by Shenzhen Microelectronics.

Shenzhen Microelectronics depends on Taiwan Silicon Corp
for semiconductor wafers.

Taiwan Silicon Corp operates a facility in Hsinchu.

The Hsinchu facility was affected by an earthquake.

Current estimated production delay is 14 days.
"""

In [2]:
from neo4j import GraphDatabase

In [3]:
NEO4J_URI="neo4j+s://50a7e94a.databases.neo4j.io"
NEO4J_USERNAME="50a7e94a"
NEO4J_PASSWORD="AVyol_j0YbCk2lfdPfldxmELehbFcVLjs5Khn973sxo"

In [4]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [5]:
document_data = {
    "document":"DOC001",
    "title": "Apex Motion's Pune plant manufactures BMS Controller X1.",
    "source": "Apex Motion's Pune plant",
    "text": document
}

In [6]:
document_data

{'document': 'DOC001',
 'title': "Apex Motion's Pune plant manufactures BMS Controller X1.",
 'source': "Apex Motion's Pune plant",
 'text': "\nApex Motion's Pune plant manufactures BMS Controller X1.\n\nBMS Controller X1 requires MCU-900.\n\nMCU-900 is supplied by Shenzhen Microelectronics.\n\nShenzhen Microelectronics depends on Taiwan Silicon Corp\nfor semiconductor wafers.\n\nTaiwan Silicon Corp operates a facility in Hsinchu.\n\nThe Hsinchu facility was affected by an earthquake.\n\nCurrent estimated production delay is 14 days.\n"}

In [7]:
def run_query(query, parameters=None):
    records, summary, keys = driver.execute_query(query, parameters_=parameters or {})
    return [record.data() for record in records]

In [8]:
query="""create (d:Document {document:$document, title:$title, source:$source, text:$text})"""
run_query(query, parameters=document_data)

[]

In [10]:
chunks = [
    {
        "chunk_id": "C001",
        "text": "Apex Motion's Pune plant manufactures BMS Controller X1."
    },
    {
        "chunk_id": "C002",
        "text": "BMS Controller X1 requires MCU-900."
    },
    {
        "chunk_id": "C003",
        "text": "MCU-900 is supplied by Shenzhen Microelectronics."
    },
    {
        "chunk_id": "C004",
        "text": "Shenzhen Microelectronics depends on Taiwan Silicon Corp for semiconductor wafers."
    },
    {
        "chunk_id": "C005",
        "text": "Taiwan Silicon Corp operates a facility in Hsinchu."
    },
    {
        "chunk_id": "C006",
        "text": "The Hsinchu facility was affected by an earthquake. Current estimated production delay is 14 days."
    }
]

query= """create (c:Chunk {chunk_id:$chunk_id, text:$text})"""
for chunk in chunks:
    run_query(query, parameters=chunk)

In [12]:
query = """MATCH (d:Document {document:$document})
MATCH (c:Chunk {chunk_id:$chunk_id})
MERGE (d)-[:HAS_CHUNK]->(c)
"""
for chunk in chunks:
    run_query(query, parameters={"document": "DOC001", "chunk_id": chunk["chunk_id"]})
